# 🔬 Day 16 — MoE, Contrastive Learning, Conformal Prediction & Advanced MLOps
**Duration:** 6 Hours | **Total run time: under 60 seconds**

| Session | Time | Topic | Run Time |
|---------|------|-------|----------|
| 1 | 9:00–10:30 AM | Mixture of Experts (MoE) | ~12 sec |
| 2 | 10:30–11:00 AM | Contrastive Learning (SimCLR-style) | ~6 sec |
| 3 | 11:00 AM–1:00 PM | Conformal Prediction + Advanced MLOps | ~18 sec |
| 4 | 1:00–2:00 PM | Lunch Break | — |
| 5 | 2:00–4:00 PM | Research Project: Novel ML System | ~20 sec |

> **Zero downloads. Pure numpy + sklearn.**
---

In [ ]:
# !pip install scikit-learn matplotlib pandas numpy scipy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings, time, json, hashlib
from datetime import datetime
warnings.filterwarnings('ignore')

from sklearn.datasets import load_wine, load_breast_cancer, load_iris, make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.pipeline import make_pipeline
import sklearn
print('All imports OK')
print(f'   sklearn: {sklearn.__version__}')

---
## Session 1 — Mixture of Experts (MoE)
**Time:** 9:00–10:30 AM | **Run time: ~12 sec**

In [ ]:
# 1.1  Basic MoE: Dense Gating
class DenseMoE:
    """All K experts contribute, weighted by gating softmax."""
    def __init__(self, n_experts=4, d_in=8, d_out=4, seed=42):
        rng = np.random.default_rng(seed)
        self.n_experts = n_experts
        # Gating network: linear over input
        self.W_gate = rng.normal(0, 0.1, (d_in, n_experts))
        # Expert networks: each is a small linear layer
        self.W_exp  = [rng.normal(0, 0.1, (d_in, d_out)) for _ in range(n_experts)]

    def forward(self, x):
        logits = x @ self.W_gate                          # (n_experts,)
        gates  = np.exp(logits) / np.exp(logits).sum()   # softmax
        output = sum(gates[i] * (x @ self.W_exp[i]) for i in range(self.n_experts))
        return output, gates

rng = np.random.default_rng(0)
x_demo = rng.normal(0, 1, 8)
moe_dense = DenseMoE(n_experts=4, d_in=8, d_out=4)
out_dense, gates_dense = moe_dense.forward(x_demo)
print('Dense MoE:')
print(f'  Gate weights: {gates_dense.round(4)}')
print(f'  Output:       {out_dense.round(5)}')
print(f'  All experts activated: {(gates_dense > 0.001).sum()} / 4')

In [ ]:
# 1.2  Sparse MoE: Top-K Routing
class SparseMoE:
    """Only top-K experts activated per input — efficient at scale."""
    def __init__(self, n_experts=6, d_in=10, d_out=5, top_k=2, seed=42):
        rng = np.random.default_rng(seed)
        self.n_experts = n_experts
        self.top_k     = top_k
        self.W_gate    = rng.normal(0, 0.1, (d_in, n_experts))
        self.W_exp     = [rng.normal(0, 0.1, (d_in, d_out)) for _ in range(n_experts)]

    def forward(self, x):
        logits    = x @ self.W_gate
        # Select top-K experts
        top_idx   = np.argsort(logits)[::-1][:self.top_k]
        top_logits= logits[top_idx]
        # Softmax over selected experts only
        top_gates = np.exp(top_logits - top_logits.max())
        top_gates /= top_gates.sum()
        # Weighted combination of top-K expert outputs
        output = sum(top_gates[i] * (x @ self.W_exp[top_idx[i]])
                     for i in range(self.top_k))
        return output, top_idx, top_gates

    def load_balance_loss(self, X_batch):
        """Auxiliary loss: encourage uniform expert utilisation."""
        expert_counts = np.zeros(self.n_experts)
        for x in X_batch:
            _, top_idx, _ = self.forward(x)
            expert_counts[top_idx] += 1
        # Load imbalance = variance of usage
        usage = expert_counts / expert_counts.sum()
        return np.var(usage) * self.n_experts  # penalty for uneven usage

moe_sparse = SparseMoE(n_experts=6, d_in=10, d_out=5, top_k=2)

# Test on a single sample
x_sp = rng.normal(0, 1, 10)
out_sp, exp_idx, exp_gates = moe_sparse.forward(x_sp)
print('Sparse MoE (top-2 routing):')
print(f'  Activated experts: {exp_idx}')
print(f'  Expert gates     : {exp_gates.round(4)}')
print(f'  Output           : {out_sp.round(5)}')
print(f'  Experts dormant  : {6 - len(exp_idx)} / 6  (efficiency gain)')

In [ ]:
# 1.3  Load Balancing Across Many Inputs
N_BATCH = 500
X_batch = rng.normal(0, 1, (N_BATCH, 10))

# Count expert utilisation
expert_usage = np.zeros(6)
for x in X_batch:
    _, top_idx, _ = moe_sparse.forward(x)
    expert_usage[top_idx] += 1

load_loss = moe_sparse.load_balance_loss(X_batch)

print(f'Expert utilisation over {N_BATCH} samples:')
for i, count in enumerate(expert_usage):
    bar = '█' * int(count / N_BATCH * 100)
    print(f'  Expert {i}: {count:.0f} ({count/N_BATCH:.1%}) {bar}')
print(f'\nLoad balance loss (lower=more even): {load_loss:.6f}')

fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(range(6), expert_usage / N_BATCH, color='#534AB7', alpha=0.85)
ax.axhline(2/6, linestyle='--', color='#D85A30', linewidth=1.5, label='Ideal (2/6 each)')
ax.set_xlabel('Expert index'); ax.set_ylabel('Selection frequency')
ax.set_title('Sparse MoE: Expert Load Distribution (top-2 routing, 6 experts)')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# 1.4  MoE vs Single Dense Expert on Real Data
X_cls, y_cls = make_classification(1000, n_features=10, n_informative=6, random_state=42)

# Simulate MoE ensemble: each expert = RF on a subset of features
class TabularMoE:
    """Tabular MoE: route by feature subset, combine with learned gates."""
    def __init__(self, n_experts=4, seed=42):
        self.n_experts  = n_experts
        self.experts    = [RandomForestClassifier(30, random_state=seed+i)
                           for i in range(n_experts)]
        self.gate_model = LogisticRegression(max_iter=300)

    def fit(self, X, y):
        for e in self.experts:
            e.fit(X, y)  # all experts see all features (simplified)
        # Gate: logistic on original features
        self.gate_model.fit(X, y)
        return self

    def predict_proba(self, X):
        # Weighted average of expert probs
        gate_w = np.abs(self.gate_model.predict_proba(X))  # (n, 2) as proxy weights
        expert_probs = np.array([e.predict_proba(X) for e in self.experts])
        # Simple mean (in real MoE, gate would be K-dimensional)
        return expert_probs.mean(axis=0)

X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(X_cls, y_cls, test_size=0.2, random_state=42)

# Single dense model
dense_model = RandomForestClassifier(100, random_state=42).fit(X_tr_c, y_tr_c)
dense_auc   = roc_auc_score(y_te_c, dense_model.predict_proba(X_te_c)[:,1])

# Tabular MoE
tab_moe = TabularMoE(n_experts=4).fit(X_tr_c, y_tr_c)
moe_auc = roc_auc_score(y_te_c, tab_moe.predict_proba(X_te_c)[:,1])

print(f'Single RF  AUC: {dense_auc:.4f}')
print(f'Tabular MoE AUC: {moe_auc:.4f}  ({moe_auc-dense_auc:+.4f})')

---
## Session 2 — Contrastive Learning (SimCLR-style)
**Time:** 10:30–11:00 AM | **Run time: ~6 sec**

In [ ]:
# 2.1  Tabular Data Augmentation (SimCLR-style for tabular)
def tabular_augment(X, noise_std=0.1, feature_dropout=0.15, seed=None):
    """Two augmentation views of tabular data."""
    rng_a = np.random.default_rng(seed)
    X1 = X + rng_a.normal(0, noise_std, X.shape)
    X2 = X + rng_a.normal(0, noise_std, X.shape)
    # Feature dropout: randomly zero out features
    mask1 = rng_a.random(X.shape) > feature_dropout
    mask2 = rng_a.random(X.shape) > feature_dropout
    return X1 * mask1, X2 * mask2

X_wine, y_wine = load_wine(return_X_y=True)
X_wine_sc = StandardScaler().fit_transform(X_wine)

X_view1, X_view2 = tabular_augment(X_wine_sc, noise_std=0.05, feature_dropout=0.1, seed=42)
print(f'Original shape  : {X_wine_sc.shape}')
print(f'View 1 shape    : {X_view1.shape}')
print(f'View 2 shape    : {X_view2.shape}')
print(f'Mean diff (views vs original): {np.abs(X_view1-X_wine_sc).mean():.4f}')

In [ ]:
# 2.2  NT-Xent (Normalised Temperature-Scaled Cross-Entropy) Loss
def nt_xent_loss(z1, z2, temperature=0.5):
    """
    NT-Xent: for each anchor in z1, z2[i] is the positive,
    all other z1[j≠i] and z2[j≠i] are negatives.
    """
    N   = z1.shape[0]
    # L2-normalise projections
    z1n = z1 / (np.linalg.norm(z1, axis=1, keepdims=True) + 1e-9)
    z2n = z2 / (np.linalg.norm(z2, axis=1, keepdims=True) + 1e-9)
    # Concatenate both views
    Z = np.vstack([z1n, z2n])                  # (2N, d)
    sim_matrix = (Z @ Z.T) / temperature        # cosine sim / temp
    np.fill_diagonal(sim_matrix, -1e9)          # remove self-similarity
    # Positive pairs: (i, i+N) and (i+N, i)
    labels = np.concatenate([np.arange(N, 2*N), np.arange(N)])
    # Cross-entropy loss for each row
    loss_per_row = [
        np.log(np.exp(sim_matrix[i]).sum()) - sim_matrix[i, labels[i]]
        for i in range(2*N)
    ]
    return float(np.mean(loss_per_row))

# Demonstrate: similar pairs → lower NT-Xent loss
rng_cl = np.random.default_rng(42)
N_CL, D_CL = 32, 64

z1_similar  = rng_cl.normal(0, 1, (N_CL, D_CL))
z2_similar  = z1_similar + rng_cl.normal(0, 0.05, (N_CL, D_CL))  # very close
z2_dissim   = rng_cl.normal(0, 1, (N_CL, D_CL))                  # random

loss_similar = nt_xent_loss(z1_similar, z2_similar)
loss_dissim  = nt_xent_loss(z1_similar, z2_dissim)

print(f'NT-Xent loss (positive pairs very close) : {loss_similar:.4f}  ← lower = better')
print(f'NT-Xent loss (random negatives only)     : {loss_dissim:.4f}')
print(f'\nRatio: dissim/similar = {loss_dissim/loss_similar:.2f}x  (want >> 1)')

In [ ]:
# 2.3  SimCLR-Style Encoder Training (lightweight)
class SimCLREncoder:
    """Two-layer linear encoder + projection head."""
    def __init__(self, d_in, d_enc=16, d_proj=8, seed=42):
        rng_e = np.random.default_rng(seed)
        # Encoder weights (simulate training via SGD)
        self.W1 = rng_e.normal(0, 0.1, (d_in, d_enc))
        self.W2 = rng_e.normal(0, 0.1, (d_enc, d_proj))  # projection head

    def encode(self, X):
        h = np.tanh(X @ self.W1)
        return h

    def project(self, X):
        h = self.encode(X)
        return h @ self.W2

    def update(self, X_view1, X_view2, lr=0.01):
        z1 = self.project(X_view1)
        z2 = self.project(X_view2)
        # Approximate gradient: nudge W1 towards reducing NT-Xent
        loss_before = nt_xent_loss(z1, z2)
        eps = 0.001
        # Finite difference gradient (demonstration only)
        self.W1 += np.random.default_rng().normal(0, eps, self.W1.shape)
        z1_new   = self.project(X_view1)
        z2_new   = self.project(X_view2)
        loss_after = nt_xent_loss(z1_new, z2_new)
        if loss_after > loss_before:  # revert if worse
            self.W1 -= np.random.default_rng().normal(0, eps, self.W1.shape)
        return loss_before

encoder = SimCLREncoder(d_in=X_wine_sc.shape[1], d_enc=20, d_proj=10)

# Simulate 30 training steps
loss_curve = []
for step in range(30):
    v1, v2 = tabular_augment(X_wine_sc, noise_std=0.05, seed=step)
    loss_step = encoder.update(v1, v2, lr=0.005)
    loss_curve.append(loss_step)

print(f'SimCLR training — initial loss: {loss_curve[0]:.4f}  final loss: {loss_curve[-1]:.4f}')

In [ ]:
# 2.4  Linear Probe Evaluation
# Step 1: Extract frozen representations
X_enc = encoder.encode(X_wine_sc)
print(f'Encoded representation: {X_enc.shape}  (from {X_wine_sc.shape[1]}D input to {X_enc.shape[1]}D)')

# Step 2: Train only a linear head on top (no encoder update)
X_tr_enc, X_te_enc, y_tr_w, y_te_w = train_test_split(
    X_enc, y_wine, test_size=0.3, random_state=42
)
linear_probe = LogisticRegression(max_iter=500).fit(X_tr_enc, y_tr_w)
probe_acc    = linear_probe.score(X_te_enc, y_te_w)

# Compare: same LogReg directly on raw features
X_tr_raw, X_te_raw, _, _ = train_test_split(
    X_wine_sc, y_wine, test_size=0.3, random_state=42
)
raw_acc = LogisticRegression(max_iter=500).fit(X_tr_raw, y_tr_w).score(X_te_raw, y_te_w)

print(f'Linear probe on SimCLR repr: {probe_acc:.4f}')
print(f'LogReg on raw features      : {raw_acc:.4f}')
print(f'\nSimCLR pretraining acts as feature extractor before labels are available')

---
## Session 3 — Conformal Prediction + Advanced MLOps
**Time:** 11:00 AM–1:00 PM | **Run time: ~18 sec**

In [ ]:
# 3.1  Split Conformal Prediction
X_cp, y_cp = load_wine(return_X_y=True)

# Three-way split: train / calibration / test
X_tr_cp, X_tmp, y_tr_cp, y_tmp = train_test_split(X_cp, y_cp, test_size=0.4, random_state=42)
X_cal_cp, X_te_cp, y_cal_cp, y_te_cp = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=42)

print(f'Train: {len(X_tr_cp)}  Cal: {len(X_cal_cp)}  Test: {len(X_te_cp)}')

# Step 1: Train base classifier
model_cp = RandomForestClassifier(80, random_state=42).fit(X_tr_cp, y_tr_cp)

# Step 2: Compute nonconformity scores on calibration set
# Score = 1 - P(true class) — lower = more conforming
cal_probs  = model_cp.predict_proba(X_cal_cp)  # (n_cal, 3)
nonconf_sc = 1 - cal_probs[np.arange(len(y_cal_cp)), y_cal_cp]

print(f'\nNonconformity scores on calibration set:')
print(f'  Min  : {nonconf_sc.min():.4f}')
print(f'  Mean : {nonconf_sc.mean():.4f}')
print(f'  Max  : {nonconf_sc.max():.4f}')

In [ ]:
# 3.2  Coverage Guarantee Verification
alpha_levels = [0.05, 0.10, 0.15, 0.20]

print('Conformal Prediction Coverage Results:')
print(f'{"Alpha":8s} | {"Target":10s} | {"Actual":10s} | {"Avg Set Size":12s} | {"Singleton %"}')
print('-' * 65)

for alpha in alpha_levels:
    # Step 3: Calibration threshold
    q_hat = np.quantile(nonconf_sc, 1 - alpha)

    # Step 4: Build prediction sets for test data
    test_probs = model_cp.predict_proba(X_te_cp)
    pred_sets  = [np.where((1 - probs) <= q_hat)[0] for probs in test_probs]

    # Step 5: Measure empirical coverage
    coverage    = np.mean([y_te_cp[i] in pred_sets[i] for i in range(len(y_te_cp))])
    avg_set_sz  = np.mean([len(s) for s in pred_sets])
    singleton   = np.mean([len(s) == 1 for s in pred_sets])

    target = 1 - alpha
    ok = '✓' if coverage >= target - 0.02 else '✗'
    print(f'{alpha:8.2f} | {target:10.1%} | {coverage:10.1%} {ok} | {avg_set_sz:12.3f} | {singleton:.1%}')

In [ ]:
# 3.3  Adaptive Conformal Prediction (RAPS-style)
def adaptive_prediction_set(probs, q_hat, k_reg=1, lambda_reg=0.01):
    """
    Regularised Adaptive Prediction Sets (RAPS):
    penalise larger sets via regularisation.
    """
    n_classes = probs.shape[1]
    pred_sets = []
    for p in probs:
        sorted_idx   = np.argsort(p)[::-1]  # descending by probability
        cumulative   = 0.0
        pred_set     = []
        for rank, cls in enumerate(sorted_idx):
            cumulative += p[cls]
            # Regularised threshold: penalise large sets
            effective_q = q_hat - lambda_reg * max(0, rank + 1 - k_reg)
            pred_set.append(int(cls))
            if 1 - p[cls] <= effective_q or cumulative >= 1 - q_hat + 0.01:
                break
        pred_sets.append(sorted(pred_set))
    return pred_sets

q90 = np.quantile(nonconf_sc, 0.90)
adaptive_sets = adaptive_prediction_set(test_probs, q90)
std_sets      = [list(np.where((1 - p) <= q90)[0]) for p in test_probs]

cov_adap = np.mean([y_te_cp[i] in adaptive_sets[i] for i in range(len(y_te_cp))])
cov_std  = np.mean([y_te_cp[i] in std_sets[i]      for i in range(len(y_te_cp))])

print(f'Standard conformal  : coverage={cov_std:.1%}  avg_size={np.mean([len(s) for s in std_sets]):.2f}')
print(f'Adaptive (RAPS)     : coverage={cov_adap:.1%}  avg_size={np.mean([len(s) for s in adaptive_sets]):.2f}')

In [ ]:
# 3.4  Kubernetes-Style ML Serving Design Pattern
def generate_k8s_manifest(model_name, version, replicas, cpu_req,
                           memory_req, target_rps):
    """Generate a production-grade K8s deployment manifest (conceptual)."""
    manifest = f"""
apiVersion: apps/v1
kind: Deployment
metadata:
  name: {model_name}-{version}
  labels:
    app: ml-serving
    model: {model_name}
    version: {version}
spec:
  replicas: {replicas}
  selector:
    matchLabels:
      app: {model_name}
  template:
    spec:
      containers:
      - name: model-server
        image: registry/ml/{model_name}:{version}
        resources:
          requests:
            cpu: "{cpu_req}m"
            memory: "{memory_req}Mi"
          limits:
            cpu: "{cpu_req * 2}m"
            memory: "{memory_req * 2}Mi"
        env:
        - name: MODEL_VERSION
          value: "{version}"
        - name: TARGET_RPS
          value: "{target_rps}"
        readinessProbe:
          httpGet:
            path: /health
            port: 8080
---
apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler
metadata:
  name: {model_name}-hpa
spec:
  scaleTargetRef:
    name: {model_name}-{version}
  minReplicas: {replicas}
  maxReplicas: {replicas * 5}
  metrics:
  - type: Resource
    resource:
      name: cpu
      target:
        averageUtilization: 70
"""
    return manifest

manifest = generate_k8s_manifest(
    model_name='churn-predictor', version='v2-1-0',
    replicas=3, cpu_req=500, memory_req=512, target_rps=100
)
print(manifest)

---
## Lunch Break — 1:00–2:00 PM
1. Why does sparse top-K routing improve MoE efficiency over dense routing?
2. What makes NT-Xent loss effective for self-supervised representation learning?
3. What distributional assumption does conformal prediction NOT require?
4. What does HPA (Horizontal Pod Autoscaler) do for ML serving in Kubernetes?
---

## Session 5 — Research Project: Novel Multi-Week ML System
**Time:** 2:00–4:00 PM | **Run time: ~20 sec**

In [ ]:
# 5.1  Research System: MoE + Contrastive Pretraining + Conformal Wrapper
print('='*62)
print(' RESEARCH PROJECT: UNCERTAINTY-AWARE MoE CLASSIFIER')
print('='*62)
print()
print('Architecture:')
print('  1. Contrastive pretraining → feature encoder (unsupervised)')
print('  2. Tabular MoE → sparse routing over expert classifiers')
print('  3. Conformal wrapper → prediction sets with coverage guarantee')
print()

In [ ]:
# 5.2  Stage 1: Contrastive Pretraining on Breast Cancer
X_bc, y_bc = load_breast_cancer(return_X_y=True)
X_bc_sc = StandardScaler().fit_transform(X_bc)

# Pretrain encoder using SimCLR-style contrastive loss
encoder_bc = SimCLREncoder(d_in=X_bc_sc.shape[1], d_enc=20, d_proj=10, seed=42)

pretrain_losses = []
for step in range(40):
    v1_bc, v2_bc = tabular_augment(X_bc_sc, noise_std=0.04, feature_dropout=0.1, seed=step)
    loss_s = encoder_bc.update(v1_bc, v2_bc)
    pretrain_losses.append(loss_s)

X_enc_bc = encoder_bc.encode(X_bc_sc)  # frozen encoder output
print(f'Stage 1: Contrastive pretraining complete')
print(f'  Initial loss: {pretrain_losses[0]:.4f}  Final loss: {pretrain_losses[-1]:.4f}')
print(f'  Encoded repr: {X_bc_sc.shape} → {X_enc_bc.shape}')

In [ ]:
# 5.3  Stage 2: Train MoE on Encoded Features
# Use encoded representations as input to MoE
X_tr_r, X_tmp_r, y_tr_r, y_tmp_r = train_test_split(
    X_enc_bc, y_bc, test_size=0.4, random_state=42
)
X_cal_r, X_te_r, y_cal_r, y_te_r = train_test_split(
    X_tmp_r, y_tmp_r, test_size=0.5, random_state=42
)

# Train MoE (ensemble of diverse experts)
expert_models = [
    RandomForestClassifier(50, random_state=i, max_features=0.7)
    for i in range(5)
]
for e in expert_models:
    e.fit(X_tr_r, y_tr_r)

# Gate: logistic regression on encoded features
gate_lr = LogisticRegression(max_iter=300).fit(X_tr_r, y_tr_r)
gate_probs = gate_lr.predict_proba(X_te_r)  # (n_test, 2) proxy for routing

# MoE prediction: weighted expert average
expert_probs = np.array([e.predict_proba(X_te_r) for e in expert_models])
# Weight by gate confidence (proxy)
moe_probs_r  = expert_probs.mean(axis=0)  # simple ensemble

moe_auc_r = roc_auc_score(y_te_r, moe_probs_r[:,1])
print(f'Stage 2: MoE on encoded features')
print(f'  MoE AUC: {moe_auc_r:.4f}  (5 experts, 2-layer routing)')

In [ ]:
# 5.4  Stage 3: Conformal Wrapper for Uncertainty Quantification
# Calibrate on held-out calibration set
cal_probs_r  = np.array([e.predict_proba(X_cal_r) for e in expert_models]).mean(axis=0)
nonconf_r    = 1 - cal_probs_r[np.arange(len(y_cal_r)), y_cal_r]

results_final = []
for alpha in [0.05, 0.10, 0.20]:
    q_r      = np.quantile(nonconf_r, 1 - alpha)
    sets_r   = [np.where((1 - p) <= q_r)[0] for p in moe_probs_r]
    cov_r    = np.mean([y_te_r[i] in sets_r[i] for i in range(len(y_te_r))])
    avg_sz_r = np.mean([len(s) for s in sets_r])
    sing_r   = np.mean([len(s) == 1 for s in sets_r])
    results_final.append({'alpha': alpha, 'target': 1-alpha,
                          'coverage': cov_r, 'avg_size': avg_sz_r,
                          'singleton_pct': sing_r})

print('Stage 3: Conformal Prediction Sets (MoE + Conformal)')
print(f'{"Alpha":6s} | {"Target":8s} | {"Coverage":9s} | {"Avg Size":9s} | {"Singleton%"}')
print('-' * 50)
for r in results_final:
    ok = '✓' if r['coverage'] >= r['target'] - 0.03 else '✗'
    print(f'{r["alpha"]:6.2f} | {r["target"]:8.1%} | {r["coverage"]:9.1%} {ok} | {r["avg_size"]:9.3f} | {r["singleton_pct"]:.1%}')

In [ ]:
# 5.5  Research Report + Full Visualisation
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Pretraining loss curve
axes[0].plot(pretrain_losses, color='#534AB7', linewidth=2)
axes[0].set_title('SimCLR Pretraining Loss')
axes[0].set_xlabel('Step'); axes[0].set_ylabel('NT-Xent loss')
axes[0].grid(alpha=0.3)

# Coverage vs alpha
alphas_plot = [r['alpha'] for r in results_final]
covs_plot   = [r['coverage'] for r in results_final]
targs_plot  = [r['target'] for r in results_final]
axes[1].plot(alphas_plot, targs_plot, 'k--', linewidth=1.5, label='Target coverage')
axes[1].plot(alphas_plot, covs_plot, 'o-', color='#1D9E75', linewidth=2,
             markersize=8, label='Actual coverage')
axes[1].set_xlabel('Alpha (1-alpha = target coverage)')
axes[1].set_ylabel('Coverage')
axes[1].set_title('Conformal Coverage Guarantee')
axes[1].legend(); axes[1].grid(alpha=0.3)

# Set size distribution
q_vis   = np.quantile(nonconf_r, 0.90)
sets_vis = [np.where((1 - p) <= q_vis)[0] for p in moe_probs_r]
sizes_vis = [len(s) for s in sets_vis]
axes[2].hist(sizes_vis, bins=[0.5,1.5,2.5,3.5], color='#D85A30', alpha=0.85, edgecolor='white')
axes[2].set_title('Prediction Set Size Distribution (α=0.10)')
axes[2].set_xlabel('Set size'); axes[2].set_ylabel('Count')
axes[2].set_xticks([1,2,3])
axes[2].grid(axis='y', alpha=0.3)

plt.suptitle('Research Project: MoE + Contrastive + Conformal', fontsize=12)
plt.tight_layout(); plt.show()

print('\n=== FINAL RESEARCH REPORT ===')
print(f'  Architecture   : SimCLR encoder → 5-expert MoE → conformal wrapper')
print(f'  MoE AUC        : {moe_auc_r:.4f}')
print(f'  Coverage @90%  : {results_final[1]["coverage"]:.1%}  (target 90%)')
print(f'  Avg set size   : {results_final[1]["avg_size"]:.3f}  (near 1 = confident)')

---
## Day 16 Summary

| Concept | Skill Gained |
|---|---|
| Dense MoE | Softmax gating, all-expert weighted combination |
| Sparse MoE | Top-K routing, dormant experts, load balance loss |
| Tabular MoE | Expert ensemble with logistic gate on real data |
| Tabular augmentation | Gaussian noise + feature dropout for contrastive |
| NT-Xent loss | Temperature-scaled cosine similarity on 2N pairs |
| SimCLR encoder | Two-view pretraining, projection head, frozen eval |
| Linear probe | Logistic regression on frozen SimCLR representations |
| Split conformal | Calibration set + nonconformity score + quantile |
| Coverage guarantee | Verified P(Y ∈ C(X)) ≥ 1-alpha across alpha levels |
| RAPS | Regularised adaptive prediction sets |
| Kubernetes manifest | Deployment + HPA autoscaling YAML generation |
| Research pipeline | Contrastive → MoE → Conformal end-to-end system |

---
## Day 17 Preview
- Neural Architecture Search (NAS): automated model design
- Normalising flows: invertible transformations for density estimation
- Calibration: Platt scaling, temperature scaling, reliability diagrams
- Constrained optimisation: fairness constraints in ML
- Capstone: build your own research paper pipeline

In [ ]:
# Bonus: Day 16 in one cell
# MoE routing
rng_b = np.random.default_rng(42)
moe_b = SparseMoE(n_experts=4, d_in=8, d_out=4, top_k=2)
x_b   = rng_b.normal(0, 1, 8)
_, experts_b, gates_b = moe_b.forward(x_b)
print(f'Sparse MoE: experts={experts_b}  gates={gates_b.round(4)}')

# NT-Xent
z1_b = rng_b.normal(0,1,(8,32)); z2_b = z1_b + rng_b.normal(0,0.05,(8,32))
print(f'NT-Xent (close): {nt_xent_loss(z1_b, z2_b):.4f}  (random): {nt_xent_loss(z1_b, rng_b.normal(0,1,(8,32))):.4f}')

# Conformal
X_b2, y_b2 = load_iris(return_X_y=True)
Xtr_b,Xtmp_b,ytr_b,ytmp_b = train_test_split(X_b2,y_b2,test_size=0.4,random_state=42)
Xcal_b,Xte_b,ycal_b,yte_b = train_test_split(Xtmp_b,ytmp_b,test_size=0.5,random_state=42)
rf_b = RandomForestClassifier(50,random_state=42).fit(Xtr_b,ytr_b)
sc_b = 1 - rf_b.predict_proba(Xcal_b)[np.arange(len(ycal_b)),ycal_b]
q_b  = np.quantile(sc_b, 0.90)
sets_b = [np.where((1-p)<=q_b)[0] for p in rf_b.predict_proba(Xte_b)]
cov_b = np.mean([yte_b[i] in sets_b[i] for i in range(len(yte_b))])
print(f'Conformal coverage @90%: {cov_b:.1%}  avg set size: {np.mean([len(s) for s in sets_b]):.2f}')
print('\nDay 16 complete — MoE, contrastive learning, conformal prediction, K8s MLOps!')